# <center>  **<span style="font-size:80px;">Exploración de los Datos (GVA)</span>** </center>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys
from src.config import Paths
Paths.init_project()

## 1. Carga de Datos Base (AMAEM + INE)

Dado que el cálculo de viviendas ilegales requiere conocer primero la estimación del INE, partimos del dataset generado en el paso del procesador de Turismo (`step2_ine.csv`).

In [ ]:
from src.water2fraud.features.amaem_processor import AMAEMProcessor
from src.water2fraud.features.ine_tourism_processor import INETourismProcessor

df_amaem = AMAEMProcessor.process(pd.read_csv(Paths.RAW_CSV_AMAEM))
df_ine   = INETourismProcessor.process(df_amaem)

## 2. Generalitat Valenciana (GVA)
### Procesamiento de Viviendas Turísticas y Hoteles

El `GVAProcessor` evalúa la línea de tiempo de altas y bajas del registro oficial para calcular el stock vivo de la ciudad mes a mes.

In [ ]:
from src.water2fraud.features.gva_processor import GVAProcessor

df_gva = GVAProcessor.process(df_ine)
display(df_gva.head())

### Ingeniería de Características (Feature Engineering)
Repartimos el total provincial registrado por la GVA hacia los barrios utilizando los porcentajes y pesos estimados previamente por el INE, lo que nos permite comparar peras con peras y calcular la métrica de ilegalidad derivada.

In [ ]:
from src.water2fraud.features.preprocessor import WaterPreprocessor

df_engineered = WaterPreprocessor._engineer_features(df_gva)
display(df_engineered[[DatasetKeys.BARRIO, DatasetKeys.FECHA, 
                       DatasetKeys.NUM_VT_BARRIO_INE, 
                       DatasetKeys.NUM_VT_BARRIO_GVA, 
                       DatasetKeys.NUM_VT_SIN_REGISTRAR]].head())

## 3. Comparativa Visual: INE vs GVA (Gap de Ilegalidad)

In [ ]:
def plot_ine_vs_gva(df, barrio=None):
    """
    Grafica la comparativa temporal de Viviendas Turísticas estimadas por el INE
    frente al registro oficial de la GVA, resaltando el GAP de viviendas ilegales.
    """
    plt.figure(figsize=(14, 6))
    sns.set_theme(style="whitegrid", context="notebook")
    
    if barrio:
        df_plot = df[df[DatasetKeys.BARRIO] == barrio].copy()
        title = f"Evolución de Viviendas Turísticas en {barrio}\nINE (Estimadas) vs GVA (Oficiales)"
        # Evitamos sumar duplicados de distintos USOS para el mismo barrio/mes
        df_plot = df_plot.drop_duplicates(subset=[DatasetKeys.FECHA])
    else:
        # Agrupamos por mes sumando los datos únicos de todos los barrios
        df_plot = df.drop_duplicates(subset=[DatasetKeys.FECHA, DatasetKeys.BARRIO]).copy()
        df_plot = df_plot.groupby(DatasetKeys.FECHA)[[
            DatasetKeys.NUM_VT_BARRIO_INE, 
            DatasetKeys.NUM_VT_BARRIO_GVA, 
            DatasetKeys.NUM_VT_SIN_REGISTRAR
        ]].sum().reset_index()
        title = "Evolución de Viviendas Turísticas Total en la Ciudad\nEstimación INE vs Registros GVA (Gap de Ilegalidad)"

    df_plot = df_plot.sort_values(DatasetKeys.FECHA)
    
    fechas = df_plot[DatasetKeys.FECHA]
    ine = df_plot[DatasetKeys.NUM_VT_BARRIO_INE]
    gva = df_plot[DatasetKeys.NUM_VT_BARRIO_GVA]
    
    # Líneas principales
    plt.plot(fechas, ine, label="Estimación INE (Turismo Real Detectado)", color='royalblue', linewidth=2.5, marker='o')
    plt.plot(fechas, gva, label="Registro Oficial GVA (Legales Activas)", color='darkorange', linewidth=2.5, marker='s')
    
    # Relleno del gap (Ilegales)
    plt.fill_between(fechas, gva, ine, where=(ine > gva), color='red', alpha=0.15, 
                     label="Gap: Viviendas Turísticas Ilegales / No Registradas", interpolate=True)
    
    plt.title(title, fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Fecha (Mes/Año)", fontsize=12)
    plt.ylabel("Número de Viviendas Turísticas", fontsize=12)
    plt.legend(loc='upper left', fontsize=11)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Gráfica Global (Suma acumulada de toda la ciudad)
plot_ine_vs_gva(df_engineered)

In [ ]:
barrios_disponibles = df_engineered[DatasetKeys.BARRIO].unique()
plot_ine_vs_gva(df_engineered, barrio=barrios_disponibles[0])

In [ ]:
barrios_disponibles = df_engineered[DatasetKeys.BARRIO].unique()
plot_ine_vs_gva(df_engineered, barrio="TABARCA")